In [ ]:
!pip install -q seqeval==1.2.2 transformers==4.28.0 datasets==2.14.5 sentencepiece==0.1.99 accelerate==0.22.0

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoConfig
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
from datasets import load_dataset, load_metric, Dataset, DatasetDict

import numpy as np
import logging

logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, padding=True ,max_length=512)

    labels = []
    for i, label in enumerate(examples[f"{task}_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

def compute_metrics(p):
    global model_name, current_epoch

    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [custom_labels[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [custom_labels[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    metric_results = {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

    return metric_results

def compute_results(trainer, tokenized_ds, metric, custom_labels):
    predictions, labels, _ = trainer.predict(tokenized_ds)
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [custom_labels[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [custom_labels[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return results

## Data Loading

You can use [Label-Studio](https://labelstud.io/) to annotate the data, export JSON format, upload to the Drive files with name (ner-train.json) and follow the next section to convert it

You can download a sample of the `ner-train.json` from [here](https://drive.google.com/file/d/1BIhxlMrvL9h5JYVcckYE4kHcoOoCWjor/view?usp=sharing)

Or by running the following cell

In [ ]:
# Optional | Download ner-train.json file, or upload your own!
!gdown 1BIhxlMrvL9h5JYVcckYE4kHcoOoCWjor

In [ ]:
import json
import itertools

In [ ]:
with open("./ner-train.json") as src:
    ner_annotated_data = json.loads(src.read())

In [ ]:
all_ner_tags, all_ner_tokens = [], []
o_tag = "O"

for rec in ner_annotated_data:
    ner_tags, ner_tokens = [], []

    text = rec["data"]["text"]
    if len(rec["annotations"]) == 0 or "result" not in rec["annotations"][0] or len(rec["annotations"][0]["result"]) == 0:
        continue

    # sort rec["annotations"][0]["result"] based on start index
    rec["annotations"][0]["result"].sort(key=lambda x: x["value"]["start"])

    # collect all ranges with their labels
    ranges = []
    for r in rec["annotations"][0]["result"]:
        ranges.append((range(r["value"]["start"], r["value"]["end"]+1), r["value"]["labels"][0]))

    # split text into tokens
    tokens = text.split()
    token_ranges = []
    c = 0
    for i, token in enumerate(tokens):
        token_ranges.append( (range(c, c+len(token)), token) )
        c += len(token) + 1

    # find all tokens that are in the ranges
    for token_range in token_ranges:
        is_found = False

        for sub_range in ranges:
            if all(e in sub_range[0] for e in token_range[0]):
                ner_tags.append(sub_range[1])
                ner_tokens.append(token_range[1])
                is_found = True
                break

        # not found
        if not is_found:
            ner_tags.append(o_tag)
            ner_tokens.append(token_range[1])

    # format BI prefix
    for i, tag in enumerate(ner_tags):
        if i == 0 and ner_tags[i] != o_tag:
            ner_tags[i] = f"B-{ner_tags[i]}"
            continue

        if i == 0 or ner_tags[i] == o_tag:
            continue

        if ner_tags[i-1].replace("B-","").replace("I-","") == ner_tags[i]:
            ner_tags[i] = f"I-{ner_tags[i]}"
        else:
            ner_tags[i] = f"B-{ner_tags[i]}"


    all_ner_tags.append(ner_tags)
    all_ner_tokens.append(ner_tokens)

In [ ]:
sample_id = 1
for tag, token in zip(all_ner_tags[sample_id], all_ner_tokens[sample_id]):
    print(tag, "--->", token)

In [ ]:
len(all_ner_tags)

10

In [ ]:
train_texts = all_ner_tokens[:8]
train_tags = all_ner_tags[:8]

dev_texts = all_ner_tokens[8:]
dev_tags = all_ner_tags[8:]

In [ ]:
set(itertools.chain.from_iterable(all_ner_tags))

# Prepare the Dataset

Next, you are going to prepare the dataset for the fine-tuning phase.

You've to provide a list of all of the NER labels in the `custom_labels` list.

We're following the popular `BIO` format. Where `O` means `not-tagged`, while any tag should represented by two labels, one for the Beggining of the labelling like `B-person`, while the other for tagging any following labelled word like `I-Person`.

Example:
If we have an example:

`I went to United States and Brazil last week`

And we have two tags for `location` and `time`

The tagged example will looks like

I `(O)` went `(O)` to `(O)` United `(B-location)` States `(I-location)` and `(O)` Brazil `(B-location)` last `(B-time)` week `(I-time)`

if we want to represent it in our training data, we'll seperate the texts and tags into different two lists like this

```
train_texts = [
    ['I', 'went', 'to', 'United', 'Stated', 'and', 'Brazil', 'last', 'week'],
    # ['anther', 'example', 'words']
]

train_tags = [
    ['O', 'O', 'O', 'B-location', 'I-location', 'O', 'B-location', 'B-time', 'I-time'],
    # ['O', 'O', 'O']
]

```

In [ ]:
# marefa-ner base checkpoint
base_checkpoint = "marefa-nlp/marefa-ner"
task = "ner"
label_all_tokens = True
seed = 101

# where to save the new model and its logs
new_model_path = f"./finetuned-ner"
logs_path = f"./logs"

# seqeval metric
metric = load_metric("seqeval")

## all of the tags in your dataset
custom_labels = ["O", "B-herb_name", "I-herb_name", "B-case", "I-case", "B-side_effect", "I-side_effect"]

device = "cuda:0"

<ipython-input-8-71d20f5d5622>:12: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric("seqeval")


In [ ]:
# train and dev text samples
# train_texts = [
#    ['برايان', 'ميلر', 'هو', 'سياسي', 'أسترالي', '،','ولد','في','30','يناير','1921','في','أستراليا','6','يونيو','2014','.'],
#    ['وقبل', 'الدخول', 'في', 'تفاصيل', 'معركتيه', 'الأخيرتين', '–', 'شبه', 'المتزامنتين', '–', 'مع', 'جون', 'برينان', 'مدير', 'وكالة', 'الاستخبارات', 'المركزية', '(CIA', ')', 'والموظفة', 'السابقة', 'لدى', 'البيت', 'الأبيض', 'وحملته', 'الانتخابية', 'الرئاسية', 'أوماروسا', 'مانيغولت', 'نيومان', 'ترامب', 'في', 'تنفيذ', 'حملته', 'الانتخابية', 'ومن', 'ثم', 'أجندته', 'الرئاسية', '.'],
# ]

# dev_texts = [
#     ['ماساكي', 'فوجيتا', '(بالكانا:ふじた', 'まさあき)', 'هو', 'لاعب', 'كرة', 'قدم', 'و', 'سياسي', 'ياباني', '،', 'ولد', 'في', '3', 'يناير', '1922', 'في', 'اليابان', '27', 'مايو', '1996', '.'],
#     ['توماس', 'مور', 'هو', 'نقابي', 'و', 'سياسي', 'أسترالي', '،', 'ولد', 'في', '14', 'فبراير', '1881', 'في', 'أستراليا', '،', 'وتوفي', 'في', '13', 'يناير', '1961', 'أستراليا', '.'],
# ]


In [ ]:
# train and dev tags
# train_tags = [
#     ['B-person','I-person','O','B-job','I-job','O','O','O','B-time','I-time','I-time','O','B-location','I-location','I-location','I-location','O'],
#     ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-person', 'I-person', 'B-job', 'B-organization', 'I-organization', 'I-organization', 'I-organization', 'O', 'O', 'O', 'O', 'B-location', 'I-location', 'O', 'O', 'O', 'B-person', 'I-person', 'I-person', 'I-person', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
# ]

# dev_tags = [
#     ['B-person', 'I-person', 'O', 'O', 'O', 'B-job', 'I-job', 'I-job', 'O', 'B-job', 'B-nationality', 'O', 'O', 'O', 'B-time', 'I-time', 'I-time', 'O', 'B-location', 'B-time', 'I-time', 'I-time', 'O'],
#     ['B-person', 'I-person', 'O', 'B-job', 'O', 'B-job', 'B-nationality', 'O', 'O', 'O', 'B-time', 'I-time', 'I-time', 'O', 'B-location', 'O', 'O', 'O', 'B-time', 'I-time', 'I-time', 'B-location', 'O'],
# ]

In [ ]:
## convert to Dataset
datasets = DatasetDict({
    "train": Dataset.from_dict({
        "tokens": train_texts,
        "ner_tags": [ [ custom_labels.index(r) for r in rec ] for rec in train_tags ]
    }),
    "dev": Dataset.from_dict({
        "tokens": dev_texts,
        "ner_tags": [ [ custom_labels.index(r) for r in rec ] for rec in dev_tags ]
    }),
})

In [ ]:
datasets

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 8
    })
    dev: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2
    })
})

## Fine-Tuning

In [ ]:
from transformers import set_seed

set_seed(seed)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(base_checkpoint,
                                                        num_labels=len(custom_labels),
                                                        ignore_mismatched_sizes=True).to(device)

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at marefa-nlp/marefa-ner and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([19, 1024]) in the checkpoint and torch.Size([7, 1024]) in the model instantiated
- classifier.bias: found shape torch.Size([19]) in the checkpoint and torch.Size([7]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# prepare dataset
tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [ ]:
# configure your fine-tuning process

args = TrainingArguments(
    new_model_path,
    logging_dir=logs_path,
    evaluation_strategy = "epoch",
    logging_strategy= "epoch",
    save_strategy= "no",
    learning_rate= 1e-4,
    load_best_model_at_end= False,
    per_device_train_batch_size= 4,
    per_device_eval_batch_size= 4,
    num_train_epochs= 10,
    weight_decay= 0.01,
    push_to_hub= False,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
train_result = trainer.train()

In [ ]:
## evaluate the model
dev_results = compute_results(trainer, tokenized_datasets["dev"], metric, custom_labels)

In [ ]:
dev_results

#### Save to Temp Storage

In [ ]:
# save the last trained weights
trainer.save_model(f"{new_model_path}/best")
tokenizer.add_tokens(custom_labels)
tokenizer.save_pretrained(f"{new_model_path}/best")

('./finetuned-ner/best/tokenizer_config.json',
 './finetuned-ner/best/special_tokens_map.json',
 './finetuned-ner/best/tokenizer.json')

#### Save to Your Google Drive

In [ ]:
## Do you want to save on your Google Drive storage ?
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


In [ ]:
!mkdir -p /gdrive/MyDrive/finetuned-ner-model-herbs

In [ ]:
new_model_path = "/gdrive/MyDrive/finetuned-ner-model-herbs"

trainer.save_model(f"{new_model_path}/best")
tokenizer.add_tokens(custom_labels)
tokenizer.save_pretrained(f"{new_model_path}/best")

('/gdrive/MyDrive/finetuned-ner-model-herbs/best/tokenizer_config.json',
 '/gdrive/MyDrive/finetuned-ner-model-herbs/best/special_tokens_map.json',
 '/gdrive/MyDrive/finetuned-ner-model-herbs/best/tokenizer.json')

## Test Your new Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

import numpy as np
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize

def _extract_ner(text: str, model: AutoModelForTokenClassification,
                 tokenizer: AutoTokenizer, start_token: str="▁"):
    tokenized_sentence = tokenizer([text], padding=True, truncation=True, return_tensors="pt")
    tokenized_sentences = tokenized_sentence['input_ids'].numpy()

    with torch.no_grad():
        output = model(**tokenized_sentence.to("cuda:0"))

    last_hidden_states = output[0].cpu().numpy()
    label_indices = np.argmax(last_hidden_states[0], axis=1)
    tokens = tokenizer.convert_ids_to_tokens(tokenized_sentences[0])
    special_tags = set(tokenizer.special_tokens_map.values())

    grouped_tokens = []
    for token, label_idx in zip(tokens, label_indices):
        if token not in special_tags:
            if not token.startswith(start_token) and len(token.replace(start_token,"").strip()) > 0:
                grouped_tokens[-1]["token"] += token
            else:
                grouped_tokens.append({"token": token, "label": custom_labels[label_idx]})

    # extract entities
    ents = []
    prev_label = "O"
    for token in grouped_tokens:
        label = token["label"].replace("I-","").replace("B-","")
        if token["label"] != "O":

            if label != prev_label:
                ents.append({"token": [token["token"]], "label": label})
            else:
                ents[-1]["token"].append(token["token"])

        prev_label = label

    # group tokens
    ents = [{"token": "".join(rec["token"]).replace(start_token," ").strip(), "label": rec["label"]}  for rec in ents ]

    return ents

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
new_model_path = f"./finetuned-ner"
# new_model_path = "/gdrive/MyDrive/finetuned-ner-model-herbs"

device = "cuda:0"

custom_labels = ["O", "B-herb_name", "I-herb_name", "B-case", "I-case", "B-side_effect", "I-side_effect"]

model_cp = f"{new_model_path}/best"

tokenizer = AutoTokenizer.from_pretrained(model_cp)
model = AutoModelForTokenClassification.from_pretrained(model_cp, num_labels=len(custom_labels)).to(device)

In [ ]:
sample = "تعتبر نبته المرمرية من النباتات المفيدة لتجنب آلام البطن و حصوات الكلية"

ents = _extract_ner(text=sample, model=model, tokenizer=tokenizer, start_token="▁")

print(ents)